**LAB 7: Fine-Tuning BERT for QA / Sentiment Tasks**

**PART 1: BERT for Sentiment Classification**

*Step 1: Install & Import*

In [1]:
!pip install transformers datasets torch

import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader

*Step 2: Sample Dataset*

In [2]:
texts = [
    "I love this movie",
    "This is terrible",
    "Amazing experience",
    "Worst acting ever"
]

labels = [1, 0, 1, 0]   # 1 = positive, 0 = negative

*Step 3: Tokenization*

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
labels = torch.tensor(labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

*Step 4: Load BERT Model*

In [4]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


*Step 5: Training Setup*

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

*Step 6: Training Loop*

In [6]:
model.train()

for epoch in range(3):
    outputs = model(**encodings, labels=labels)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7377
Epoch 2, Loss: 0.5737
Epoch 3, Loss: 0.5499


*Step 7: Prediction*

In [7]:
model.eval()

test_text = ["I really enjoyed this"]
inputs = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)

outputs = model(**inputs)
pred = torch.argmax(outputs.logits, dim=1)

print("Positive" if pred.item()==1 else "Negative")

Negative


**PART 2: BERT for Question Answering**

*Step 8: Load QA Model*

In [11]:
from transformers import BertForQuestionAnswering

qa_model = BertForQuestionAnswering.from_pretrained(
    'bert-large-uncased-whole-word-masking-finetuned-squad'
)

config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


*Step 9: Input Question + Context*

In [12]:
question = "Where do I live?"
context = "My name is Sakshi and I live in India."

inputs = tokenizer(question, context, return_tensors="pt")

*Step 10: Predict Answer*

In [13]:
outputs = qa_model(**inputs)

start = torch.argmax(outputs.start_logits)
end = torch.argmax(outputs.end_logits)

# Fix invalid span
if end < start:
    end = start + 1
else:
    end = end + 1

answer = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
)

print("Answer:", answer)

Answer: india
